<!-- SPDX-License-Identifier: Apache-2.0 -->
<!-- Copyright 2026 Evan Gress -->

# 02 · Pipe flow, property tables, and root finding

Three tools you will use constantly:
1. **pandas** to load and inspect reference tables
2. **interpolation** to read between table rows
3. **root finding** to solve equations that have no algebraic solution

## 1. Load a table with pandas

In [ ]:
from fpc import Properties, Fluids

props = Properties()
props.list_tables()

In [ ]:
water = props.load_table(name="water_liquid_1atm")
water.head()

In [ ]:
water.describe().T[["min", "max"]]

Boolean masks select rows — *which temperatures have viscosity below 0.5 mPa·s?*

In [ ]:
water[water["dynamic_viscosity_pa_s"] < 0.5e-3][["temperature_c", "dynamic_viscosity_pa_s"]]

## 2. Interpolate between rows
The table jumps in 5–10 °C steps. `water_liquid` interpolates; `method="cubic"` uses a smooth spline instead of straight lines. Outside the table it raises an error rather than guessing.

In [ ]:
props.water_liquid(temperature_c=37, property="dynamic_viscosity_pa_s")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

sat = props.load_table(name="water_saturation")
t = np.linspace(0.01, 200, 400)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(sat["temperature_c"], sat["pressure_kpa"], "o", label="table")
ax.plot(t, props.water_saturation(temperature_c=t, property="pressure_kpa"), label="linear")
ax.plot(t, props.water_saturation(temperature_c=t, property="pressure_kpa", method="cubic"), "--", label="cubic")
ax.set_xlabel("Temperature [degC]"); ax.set_ylabel("Saturation pressure [kPa]")
ax.legend(); ax.grid(alpha=0.3)

## 3. Root finding: the Colebrook equation
$$\frac{1}{\sqrt f} = -2\log_{10}\left(\frac{\varepsilon/D}{3.7} + \frac{2.51}{Re\sqrt f}\right)$$
`f` is on both sides, so we can't rearrange it. `Fluids.colebrook_friction_factor` hands the equation to `scipy.optimize.brentq`. Repeating that solve over a grid draws a Moody chart:

In [ ]:
fluids = Fluids()
reynolds = np.logspace(3, 8, 200)
fig, ax = plt.subplots(figsize=(6, 4))
for roughness in (0, 1e-5, 1e-4, 1e-3, 1e-2):
    f = [fluids.colebrook_friction_factor(reynolds=re, relative_roughness=roughness) for re in reynolds]
    ax.loglog(reynolds, f, label=f"eps/D = {roughness:g}")
ax.set_xlabel("Reynolds number"); ax.set_ylabel("Darcy friction factor")
ax.legend(fontsize=8); ax.grid(which="both", alpha=0.3)

## 4. Put it together
`Fluids.pipe_flow` chains all of this for one pipe:

In [ ]:
rho = props.water_liquid(temperature_c=20, property="density_kg_m3")
mu = props.water_liquid(temperature_c=20, property="dynamic_viscosity_pa_s")
fluids.pipe_flow(flow=0.012, diameter=0.1023, length=120, roughness=0.045e-3, density=rho, viscosity=mu)

**Try it:** how much does the pressure drop fall if the water is at 80 °C instead of 20 °C? Why?